## Load RNN and compare against HMM at forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import rpy2.robjects as robjects

In [ ]:
from custom_losses import (
    dirichlet_layer,
    EvidentialLoss,
    evidential_kl_divergence,
    cross_entropy_loss,
)

In [ ]:
readRDS = robjects.r['readRDS']

def load_rds_array(path):
    """Load an R array into numpy, preserving shape and dimnames."""
    r_obj    = readRDS(path)
    dims     = tuple(int(d) for d in robjects.r['dim'](r_obj))
    arr      = np.array(r_obj).reshape(dims, order='F')
    r_dimnames = robjects.r['dimnames'](r_obj)
    dimnames = []
    for dn in r_dimnames:
        if dn == robjects.rinterface.NULL:
            dimnames.append(None)
        else:
            dimnames.append(list(dn))
    return arr, dimnames

X_train,     (_, _, feature_names)   = load_rds_array('energy_data/X_train.rds')
Y_train,     (_, _, state_names)     = load_rds_array('energy_data/Y_train.rds')
Y_hat_train, _                       = load_rds_array('energy_data/Y_hat_train.rds')

X_valid,     _ = load_rds_array('energy_data/X_valid.rds')
Y_valid,     _ = load_rds_array('energy_data/Y_valid.rds')
Y_hat_valid, _ = load_rds_array('energy_data/Y_hat_valid.rds')

X_test,      _ = load_rds_array('energy_data/X_test.rds')
Y_test,      _ = load_rds_array('energy_data/Y_test.rds')
Y_hat_test,  _ = load_rds_array('energy_data/Y_hat_test.rds')

print("X_train:    ", X_train.shape)
print("Y_train:    ", Y_train.shape)
print("Feature names:", feature_names)
print("State names:  ", state_names)

In [ ]:
# HMM transition matrix
transition_matrix = pd.read_csv('energy_data/transition_matrix.csv', index_col=0)
transition_matrix.columns = range(len(transition_matrix.columns))
transition_matrix.index   = range(len(transition_matrix.index))
print("Transition matrix:")
print(transition_matrix)

In [ ]:
best_model = tf.keras.models.load_model('energy_data/best_model.keras')
best_model.summary()

# Extract ground truth (drop date column)
y_cols = [i for i, n in enumerate(state_names) if n != 'date']
Y_test_gt   = Y_test[:, :, y_cols].astype(float)      # (n_samples, 10, 3)
Y_hat_hmm   = Y_hat_test[:, :, y_cols].astype(float)   # (n_samples, 10, 3)

# Get RNN predictions — reshape horizon-split back to (n_samples, horizon, states)
forecast_horizon = Y_test.shape[1]
n_states         = len(y_cols)

# Prepare test inputs the same way as notebook 3
dt_col_names = ['month', 'day_of_week']
x_cols   = [i for i, n in enumerate(feature_names) if n != 'date' and n not in dt_col_names]
x_dt_cols = [i for i, n in enumerate(feature_names) if n in dt_col_names]
x_date_idx = feature_names.index('date')
y_date_idx = state_names.index('date')

X_test_cov = np.concatenate([X_test[:, :, x_cols].astype(float)] * forecast_horizon, axis=0)
X_test_dt  = np.concatenate([X_test[:, :, x_dt_cols].astype(float)] * forecast_horizon, axis=0)
horizon_test = np.repeat(np.eye(forecast_horizon), X_test.shape[0], axis=0)

rnn_preds_flat = best_model.predict([X_test_cov, X_test_dt, horizon_test])
rnn_probs_flat, _, _ = dirichlet_layer(tf.constant(rnn_preds_flat, dtype=tf.float32))
rnn_probs_flat = rnn_probs_flat.numpy()

n_samples = X_test.shape[0]
Y_hat_rnn = rnn_probs_flat.reshape(forecast_horizon, n_samples, n_states).transpose(1, 0, 2)

print("Y_test_gt: ", Y_test_gt.shape)
print("Y_hat_hmm: ", Y_hat_hmm.shape)
print("Y_hat_rnn: ", Y_hat_rnn.shape)

In [ ]:
# Cross-entropy loss per sample per horizon
def ce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=-1)

hmm_loss = ce_loss(Y_test_gt, Y_hat_hmm)  # (n_samples, 10)
rnn_loss = ce_loss(Y_test_gt, Y_hat_rnn)  # (n_samples, 10)

# --- Plot 1: Mean loss by forecast horizon ---
fig, ax = plt.subplots(figsize=(8, 5))
horizons = np.arange(1, forecast_horizon + 1)
ax.plot(horizons, hmm_loss.mean(axis=0), 'o-', label='HMM', linewidth=2)
ax.plot(horizons, rnn_loss.mean(axis=0), 's-', label='RNN', linewidth=2)
ax.set_xlabel('Forecast Horizon (days ahead)')
ax.set_ylabel('Mean Cross-Entropy Loss')
ax.set_title('HMM vs RNN: Loss by Forecast Horizon')
ax.set_xticks(horizons)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Plot 2: Loss distribution (boxplot side-by-side per horizon) ---
fig, ax = plt.subplots(figsize=(12, 5))
positions_hmm = horizons - 0.2
positions_rnn = horizons + 0.2

bp_hmm = ax.boxplot(hmm_loss, positions=positions_hmm, widths=0.3,
                     patch_artist=True, showfliers=False,
                     boxprops=dict(facecolor='#1f77b4', alpha=0.6),
                     medianprops=dict(color='black'))
bp_rnn = ax.boxplot(rnn_loss, positions=positions_rnn, widths=0.3,
                     patch_artist=True, showfliers=False,
                     boxprops=dict(facecolor='#ff7f0e', alpha=0.6),
                     medianprops=dict(color='black'))

ax.set_xlabel('Forecast Horizon (days ahead)')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('HMM vs RNN: Loss Distribution by Horizon')
ax.set_xticks(horizons)
ax.legend([bp_hmm['boxes'][0], bp_rnn['boxes'][0]], ['HMM', 'RNN'])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Recover total evidence S per prediction
_, _, rnn_S_flat = dirichlet_layer(tf.constant(rnn_preds_flat, dtype=tf.float32))
rnn_test_S = rnn_S_flat.numpy().reshape(forecast_horizon, n_samples).T  # (n_samples, 10)

# Scatter: RNN loss colored by evidence, HMM boxplot as reference
fig, axes = plt.subplots(1, forecast_horizon, figsize=(20, 5), sharey=True)

for h in range(forecast_horizon):
    ax = axes[h]

    # HMM boxplot as background reference
    ax.boxplot(hmm_loss[:, h], positions=[0], widths=0.6,
               patch_artist=True, showfliers=False,
               boxprops=dict(facecolor='#1f77b4', alpha=0.3),
               medianprops=dict(color='black'))

    # RNN scatter colored by evidence S
    sc = ax.scatter(
        np.random.normal(1, 0.08, n_samples),  # jitter x position
        rnn_loss[:, h],
        c=rnn_test_S[:, h],
        cmap='RdYlGn',  # red=low evidence, green=high evidence
        s=6, alpha=0.5,
        vmin=np.percentile(rnn_test_S, 5),
        vmax=np.percentile(rnn_test_S, 95),
    )

    ax.set_title(f'h={h+1}', fontsize=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['HMM', 'RNN'], fontsize=8)
    if h == 0:
        ax.set_ylabel('Cross-Entropy Loss')

fig.colorbar(sc, ax=axes, orientation='vertical', fraction=0.015, pad=0.02, label='RNN Evidence (S)')
fig.suptitle('HMM vs RNN Loss by Horizon — RNN colored by model confidence', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
confidence_threshold = 18

# --- Plot 1: Mean loss by horizon (RNN filtered by S >= threshold) ---
fig, ax = plt.subplots(figsize=(8, 5))
horizons = np.arange(1, forecast_horizon + 1)

hmm_mean = np.zeros(forecast_horizon)
rnn_mean = np.zeros(forecast_horizon)
rnn_count = np.zeros(forecast_horizon)

for h in range(forecast_horizon):
    mask = rnn_test_S[:, h] >= confidence_threshold
    rnn_count[h] = mask.sum()
    hmm_mean[h] = hmm_loss[mask, h].mean()
    rnn_mean[h] = rnn_loss[mask, h].mean()

ax.plot(horizons, hmm_mean, 'o-', label='HMM (matched samples)', linewidth=2)
ax.plot(horizons, rnn_mean, 's-', label=f'RNN (S >= {confidence_threshold})', linewidth=2)
ax.set_xlabel('Forecast Horizon (days ahead)')
ax.set_ylabel('Mean Cross-Entropy Loss')
ax.set_title(f'HMM vs RNN: Loss by Horizon (RNN confidence >= {confidence_threshold})')
ax.set_xticks(horizons)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Samples per horizon:", rnn_count.astype(int))
print(f"% retained: {(rnn_count / n_samples * 100).round(1)}")

# --- Plot 2: Boxplot side-by-side (filtered) ---
fig, ax = plt.subplots(figsize=(12, 5))
positions_hmm = horizons - 0.2
positions_rnn = horizons + 0.2

hmm_filtered = [hmm_loss[rnn_test_S[:, h] >= confidence_threshold, h] for h in range(forecast_horizon)]
rnn_filtered = [rnn_loss[rnn_test_S[:, h] >= confidence_threshold, h] for h in range(forecast_horizon)]

bp_hmm = ax.boxplot(hmm_filtered, positions=positions_hmm, widths=0.3,
                     patch_artist=True, showfliers=False,
                     boxprops=dict(facecolor='#1f77b4', alpha=0.6),
                     medianprops=dict(color='black'))
bp_rnn = ax.boxplot(rnn_filtered, positions=positions_rnn, widths=0.3,
                     patch_artist=True, showfliers=False,
                     boxprops=dict(facecolor='#ff7f0e', alpha=0.6),
                     medianprops=dict(color='black'))

ax.set_xlabel('Forecast Horizon (days ahead)')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title(f'HMM vs RNN: Loss Distribution (RNN confidence >= {confidence_threshold})')
ax.set_xticks(horizons)
ax.legend([bp_hmm['boxes'][0], bp_rnn['boxes'][0]], ['HMM (matched)', f'RNN (S >= {confidence_threshold})'])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()